# Weekly College Football Predictions

This is the notebook that will normally be run once per week. It will:

1. Refresh the current season's completed games and upcoming schedule
2. Recalculate all pregame features
3. Refit all six tuned base models using the latest completed games
4. Apply the saved stacking weights and selected primary systems
5. Add market and hybrid predictions where betting lines are available
6. Save a timestamped copy of the predictions

The hyperparameters are not retuned each week. This prevents the weekly process from repeatedly adapting to a small amount of new data. Previously saved predictions are never overwritten.

In [1]:
from pathlib import Path
import sys

# Allow the notebook to run from either the project root or the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Imports and Settings

In [2]:
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from src.config import (
    CURRENT_SEASON,
    EVALUATION_DIR,
    INTERIM_DIR,
    MODEL_DIR,
    PREDICTION_DIR,
    PROCESSED_DIR,
    START_SEASON,
)
from src.data import download_all_data
from src.features import build_modeling_table, build_team_games, check_for_leakage
from src.models import (
    add_prediction_intervals,
    add_score_predictions,
    load_model_bundle,
    refit_model_bundle,
)

pd.set_option('display.max_columns', 100)
RUN_TIME = datetime.now(timezone.utc)
print('Prediction time (UTC):', RUN_TIME.strftime('%Y-%m-%d %H:%M'))

Prediction time (UTC): 2026-09-30 15:01


## Refreshing the Current Season

In [3]:
tables = download_all_data(
    start_season=START_SEASON,
    end_season=CURRENT_SEASON,
    refresh_current=True,
)

team_games = build_team_games(
    games=tables['games'],
    team_stats=tables['team_stats'],
    advanced_stats=tables['advanced_stats'],
)
team_games.to_csv(INTERIM_DIR / 'team_games.csv', index=False)

modeling_table = build_modeling_table(
    games=tables['games'],
    team_games=team_games,
    talent=tables['talent'],
    recruiting=tables['recruiting'],
    lines=tables['lines'],
)
check_for_leakage(modeling_table)
modeling_table.to_csv(PROCESSED_DIR / 'modeling_table.csv', index=False)

## Selecting the Upcoming Week

In [4]:
future_games = modeling_table[
    modeling_table['completed'].eq(False) &
    modeling_table['start_date'].ge(pd.Timestamp(RUN_TIME))
].sort_values('start_date')

if future_games.empty:
    raise ValueError('No future FBS-vs-FBS games were found in the current schedule.')

next_game = future_games.iloc[0]
upcoming_games = future_games[
    future_games['season'].eq(next_game['season']) &
    future_games['season_type'].eq(next_game['season_type']) &
    future_games['week'].eq(next_game['week'])
].copy()

print('Season:', int(next_game['season']))
print('Week:', int(next_game['week']))
print('Games:', len(upcoming_games))

Season: 2026
Week: 5
Games: 56


## Refitting the Production Models

In [5]:
saved_bundle = load_model_bundle(MODEL_DIR / 'statistics_only_models.joblib')

training_games = modeling_table[
    modeling_table['completed'].eq(True) &
    modeling_table['start_date'].lt(pd.Timestamp(RUN_TIME)) &
    modeling_table['actual_margin'].notna() &
    modeling_table['actual_total'].notna()
].sort_values(['start_date', 'game_id'])

production_bundle = refit_model_bundle(training_games, saved_bundle)
print('Production training games:', len(training_games))
print('Margin system:', production_bundle['margin_primary'])
print('Total system:', production_bundle['total_primary'])

Fitting Ridge
Fitting Elastic Net


Fitting Random Forest


Fitting Gradient Boosting


Fitting RBF Support Vector Regression


Fitting Neural Network


Fitting Ridge
Fitting Elastic Net


Fitting Random Forest


Fitting Gradient Boosting


Fitting RBF Support Vector Regression


Fitting Neural Network


Production training games: 9300
Margin system: Random Forest
Total system: Stacking Ensemble


## Generating Predictions

In [6]:
predictions = add_score_predictions(upcoming_games, production_bundle)

calibration_predictions = pd.read_csv(EVALUATION_DIR / 'calibration_predictions.csv')
predictions = add_prediction_intervals(
    predictions,
    calibration_predictions,
    coverage=0.90,
)

hybrid_path = MODEL_DIR / 'hybrid_models.joblib'
if hybrid_path.exists():
    hybrid_models = joblib.load(hybrid_path)
    has_market = predictions['market_margin'].notna() & predictions['market_total'].notna()
    predictions.loc[has_market, 'hybrid_margin'] = hybrid_models['margin_hybrid'].predict(
        predictions.loc[has_market, ['model_margin', 'market_margin']]
    )
    predictions.loc[has_market, 'hybrid_total'] = hybrid_models['total_hybrid'].predict(
        predictions.loc[has_market, ['model_total', 'market_total']]
    )
    predictions.loc[has_market, 'hybrid_home_score'] = np.maximum(
        0,
        (predictions.loc[has_market, 'hybrid_total'] + predictions.loc[has_market, 'hybrid_margin']) / 2,
    )
    predictions.loc[has_market, 'hybrid_away_score'] = np.maximum(
        0,
        (predictions.loc[has_market, 'hybrid_total'] - predictions.loc[has_market, 'hybrid_margin']) / 2,
    )

## Upcoming Games

In [7]:
display_columns = [
    'start_date', 'away_team', 'home_team',
    'model_away_score', 'model_home_score',
    'model_margin_low', 'model_margin_high',
    'market_margin', 'market_total',
]
if 'hybrid_home_score' in predictions.columns:
    display_columns += ['hybrid_away_score', 'hybrid_home_score']

weekly_table = predictions[display_columns].copy()
score_columns = [column for column in weekly_table.columns if 'score' in column]
weekly_table[score_columns] = weekly_table[score_columns].round(0).astype('Int64')
weekly_table.round(1)

,start_date,away_team,home_team,model_away_score,model_home_score,model_margin_low,model_margin_high,market_margin,market_total,hybrid_away_score,hybrid_home_score
9301,2026-10-02 00:00:00+00:00,Western Kentucky,New Mexico State,31,28,-29.5,23.5,2.5,57.5,27,31
9302,2026-10-02 01:00:00+00:00,North Texas,Tulsa,34,25,-36.1,17.0,1.2,57.2,28,30
9303,2026-10-02 23:00:00+00:00,Pittsburgh,Virginia Tech,27,27,-27.3,25.7,3.0,54.8,26,29
9304,2026-10-02 23:00:00+00:00,Liberty,Delaware,30,23,-33.2,19.8,-7.0,50.2,29,23
9305,2026-10-03 00:00:00+00:00,Penn State,Northwestern,27,21,-31.9,21.1,-2.8,45.5,25,23
9315,2026-10-03 16:00:00+00:00,Navy,Air Force,23,27,-23.0,30.1,3.0,45.8,23,27
9314,2026-10-03 16:00:00+00:00,Michigan,Minnesota,23,21,-28.2,24.9,-5.5,43.2,26,21
9313,2026-10-03 16:00:00+00:00,Syracuse,UConn,21,29,-18.4,34.7,-6.5,50.5,29,23
9312,2026-10-03 16:00:00+00:00,Stanford,Wake Forest,25,36,-15.9,37.2,12.8,53.5,22,35
9311,2026-10-03 16:00:00+00:00,Notre Dame,North Carolina,32,17,-41.7,11.4,-21.0,47.2,35,15


## Saving the Prediction Archive

In [8]:
predictions['generated_at_utc'] = RUN_TIME.isoformat()
filename = (
    f"predictions_{int(next_game['season'])}_"
    f"{next_game['season_type']}_week_{int(next_game['week']):02d}_"
    f"{RUN_TIME.strftime('%Y%m%d_%H%M%S')}.csv"
)
output_path = PREDICTION_DIR / filename
predictions.to_csv(output_path, index=False)

print('Predictions saved to:', output_path.relative_to(PROJECT_ROOT))

Predictions saved to: outputs/weekly_predictions/predictions_2026_regular_week_05_20260930_150149.csv


## Interpreting the Output

`model_home_score` and `model_away_score` are produced without betting information. The model-specific columns preserve the predictions from all six base models and the ensemble for later analysis. `market_margin` and `market_total` show the available consensus line. Hybrid scores are only shown when a market line is available.

The margin interval describes uncertainty around the predicted difference between the two teams. It should not be interpreted as a guarantee that the final margin will fall inside the interval.